# 第6周 b｜Cox模型与HR

学习目标：部分似然、比例风险、基线危险、HR、临床单位。

> **本周怎么学**：先学精讲 `w06a_cox_regression.ipynb`，再做这份**工程实践**。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：说明HR不是风险比，更不是五年概率。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 从相对危险到绝对风险
Cox模型的HR描述瞬时危险的相对倍数。要得到五年风险，还需要训练集估计的基线累积危险。这里先用少量预先指定变量，理解模型接口与输出。

In [2]:
from primecvd.survival import CoxModel
features = ["Age", "BMI", "SBP", "diabetes", "smoking_status"]
model = CoxModel(features, alpha=0.0).fit(train)
display(model.coefficient_table())
print("优化检查：", model.diagnostics)

,term,log_HR,HR,SE,ci_low,ci_high,p_value
0,Age_per10,0.716443,2.047138,0.024135,1.952553,2.146304,1.209329e-193
1,BMI_per5,0.122390,1.130195,0.028469,1.068857,1.195052,1.715564e-05
2,SBP_per10,0.095962,1.100717,0.018644,1.061221,1.141683,2.645744e-07
3,diabetes,2.500430,12.187729,0.059297,10.850455,13.689817,0.000000e+00
4,smoking_ex_vs_non,0.349798,1.418781,0.066934,1.244342,1.617673,1.732288e-07
5,smoking_current_vs_non,0.496068,1.642252,0.081044,1.401049,1.924979,9.302045e-10


优化检查： {'optimizer_success': True, 'message': 'Optimization terminated successfully.', 'iterations': 35, 'max_abs_gradient': 1.188047755574319e-08, 'n': 35000, 'events': 1406, 'objective': 0.35959791401692764}


In [3]:
p = model.predict_risk(valid, cfg["horizon_years"])
display(pd.DataFrame({"Patient_ID":valid.Patient_ID.to_numpy()[:8], "predicted_risk":p[:8]}))
print("概率是否都在[0,1]：", bool(np.all((p>=0) & (p<=1))))

,Patient_ID,predicted_risk
0,416,0.007202
1,569,0.010518
2,1037,0.003362
3,1997,0.001613
4,2456,0.027146
5,5069,0.001988
6,6617,0.092812
7,7181,0.014426


概率是否都在[0,1]： True


HR表中的`Age_per10`按10岁解释，`BMI_per5`按5 kg/m²解释。分类变量参考水平写在方法说明。系数CI是给定模型的点态区间，未做多重比较调整，也不是预测性能CI。

In [4]:
residuals = model.schoenfeld_residuals(train)
display(residuals.head())
# 残差可用于探索PH假设；不是本项目已经完成了正式全局PH检验。

,event_time,Age_per10,BMI_per5,SBP_per10,diabetes,smoking_ex_vs_non,smoking_current_vs_non
0,2.963423,-0.970463,-0.386730,-1.065998,-2.169660,-0.571296,-0.420592
1,0.197023,-2.030240,1.128379,0.794797,1.260552,-0.558456,2.840407
2,2.961116,0.750235,-0.555696,-0.574076,-2.169574,-0.571273,-0.420575
3,3.539723,0.633528,0.072520,0.919369,-2.106244,-0.570997,-0.417247
4,3.800478,0.235488,0.429432,0.828291,1.740674,-0.554299,2.883768


## 独立练习
用自己的话说明HR=1.5不代表五年风险增加50个百分点。随后改用完整预先指定字段，比较少变量模型和完整模型；在打开测试集之前完成探索。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：PH残差图仅为探索，不冒充完整假设检验。